In [ ]:
from pathlib import Path

from inspect_ai.log import EvalLog, read_eval_log

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
# Resolve local path to Inspect logs
notebook_dir = Path.cwd()
log_path = notebook_dir.parent / "results" / "llama3.3-70b_indic_calibration_n50.eval"

if log_path.exists():
    print(f"Success. Found log at: {log_path}")
    log = read_eval_log(str(log_path))
else:
    print(f"Error. Could not find log at: {log_path}")
    print(f"Notebook is running in: {notebook_dir}")

In [ ]:
def extract_calibration_data(log: EvalLog) -> tuple[np.ndarray, np.ndarray, int]:
    """Extract confidence scores, accuracy values and bin counts from an Inspect EvalLog."""
    confidence_values = []
    accuracy_values = []

    for sample in log.samples:
        # Access the specific scorer result for this sample
        scorer_data = sample.scores.get("indic_calibration_scorer", None)

        if scorer_data:
            confidence = scorer_data.metadata.get("confidence", None)
            accuracy = scorer_data.value

            if confidence is not None:
                confidence_values.append(float(confidence))
                accuracy_values.append(float(accuracy))

    # Retrieve the n_bins setting used during the evaluation run
    n_bins = int(log.eval.scorers[0].metrics[0].options["n_bins"])

    return np.array(confidence_values), np.array(accuracy_values), n_bins

In [ ]:
def plot_calibration_diagram(confidence_values: np.ndarray, accuracy_values: np.ndarray, n_bins: int, model_name: str = "Llama 3.3-70b-Versatile") -> None:
    # Define bin edges and calculate centers for bar placement
    bins = np.linspace(0, 100, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    bin_accuracies = []
    bin_counts = []

    # Calculate statistics for each confidence interval
    for i in range(len(bins) - 1):
        # Identify samples falling within the current bin range [start, end)
        bin_mask = (confidence_values >= bins[i]) & (confidence_values < bins[i + 1])
        
        if np.any(bin_mask):
            bin_accuracies.append(np.mean(accuracy_values[bin_mask]))
            bin_counts.append(np.sum(bin_mask))
        else:
            bin_accuracies.append(0)
            bin_counts.append(0)
    
    sns.set_style("whitegrid")
    plt.figure(figsize=(10, 7))

    plt.bar(bin_centers, bin_accuracies, width=(100/n_bins)-2, color="#5D3FD3", edgecolor="black", alpha=0.7, hatch="//", label="Actual accuracy")

    # Plot the perfect calibration line (y = x)
    plt.plot([0, 100], [0, 1], "r--", linewidth=2, label="Perfect calibration")

    # Annotate bars with the number of samples in each bin
    for i, count in enumerate(bin_counts):
        if count > 0:
            plt.text(bin_centers[i], bin_accuracies[i] + 0.03, str(int(count)), ha="center", va="bottom", fontsize=10, fontweight="bold", bbox=dict(facecolor="black", alpha=0.8, edgecolor="none", boxstyle="round,pad=0.3"), color="white")
    
    # Add labels
    plt.xlabel("Predicted confidence (%)", fontsize=14, fontweight="bold")
    plt.ylabel("Actual accuracy", fontsize=14, fontweight="bold")
    plt.title(f"IndicCalibrationBench - Calibration (Reliability) Diagram: {model_name}", fontsize=16, fontweight="bold", pad=20)

    plt.xticks(bins)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xlim(0, 100)
    plt.ylim(0, 1.1)

    # Add legend
    plt.legend(loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()

    plt.show()

In [ ]:
confidences, accuracies, n_bins = extract_calibration_data(log)

In [ ]:
plot_calibration_diagram(confidences, accuracies, n_bins, model_name="Llama 3.3-70b-Versatile")